In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [2]:
from utils.train_utils import Evaluate_model
from utils.datautils import Readdataset, calculate_dataset_metrics, Splitview

plot the actual data

In [3]:
df = pd.read_csv('../UCRArchive_2018/SimulatedSagHealthy 2/SimulatedSagHealthy 2_TRAIN.tsv', sep='\t', header=None)
labels = df.pop(0)
df_signal = pd.DataFrame(df)
df_signal["label"] = labels
df_signal["signal_id"] = df_signal.index

# Melt the dataframe: each row represents a time point of a signal.
df_long = df_signal.melt(id_vars=["signal_id", "label"], var_name="Time", value_name="Amplitude")

# Create the line plot with signals grouped by signal_id and colored by label.
fig = px.line(df_long, x="Time", y="Amplitude", color="label", line_group="signal_id",
              title="Signals of Original Simulated Data Colored by Label")
# fig.add_hline(y=-0.0095, line_color="black")
fig.show()

Looking at NSTSC handled data

In [4]:
Dataset_name = "SimulatedSagHealthy 2"
dataset_path_ = "../UCRArchive_2018/"
normalize_dataset = True
Xtrain, ytrain, Xval, yval, Xtest, ytest = Readdataset(dataset_path_, Dataset_name)

In [5]:
file = "../Tree_Models/SimulatedSagHealthy 2_model.pkl"
with open(file, "rb") as f:
    Tree = pickle.load(f)

In [6]:
N, T = calculate_dataset_metrics(Xtrain)

In [7]:
Xraw, Xfft, Xder = Splitview(Xtrain, T)

In [8]:
print("N:", N)
print("T:", T)

N: 70
T: 614


In [9]:
testaccu = Evaluate_model(Tree, Xtest, ytest)
print("Test accuracy:", testaccu)

Test accuracy: 1.0


In [10]:
Tree

{0: <utils.train_utils.Node at 0x15da0a73cb0>,
 1: <utils.train_utils.Node at 0x15da0b13110>,
 2: <utils.train_utils.Node at 0x15da0b13250>}

In [11]:
Tree[0].bestmodel

TL_NN1()

In [23]:
Tree[0].trueidx

array([ 0,  1,  2,  5,  6,  7,  8, 10, 13, 14, 16, 18, 21, 25, 28, 29, 30,
       31, 32, 34, 35, 39, 41, 43, 45, 48, 49, 50, 52, 54, 56, 61, 62, 63,
       64, 66, 68, 69])

In [21]:
Tree[0].bstmdlclass

np.int64(0)

In [12]:
node0 = Tree[0].bestmodel.state_dict()

In [13]:
import torch.nn.functional as F
import torch

# normalizing the weights A1 to A4
A1_sm = F.softmax(node0['A1'], dim=1)
A2_sm = F.softmax(node0['A2'], dim=1)
A3_sm = F.softmax(node0['A3'], dim=1)
A4_sm = F.softmax(node0['A4'], dim=1)
# printing the top 3 weights and indices for A1 to A3
def print_top_weights(A_sm, name):
    flat_A_sm = A_sm.flatten()
    topk = torch.topk(flat_A_sm, 5)
    print(f"Top 3 weights for {name}:")
    for i in range(5):
        print(f"Weight: {topk.values[i].item()}, Index: {topk.indices[i].item()}")
    print()
print_top_weights(A1_sm, "A1_sm_0")
print_top_weights(A2_sm, "A2_sm_0")
print_top_weights(A3_sm, "A3_sm_0")
print(f"A4_sm_0: {A4_sm}\n")

Top 3 weights for A1_sm_0:
Weight: 0.0039502219296991825, Index: 87
Weight: 0.0034261629916727543, Index: 218
Weight: 0.0034015325363725424, Index: 276
Weight: 0.0033810671884566545, Index: 278
Weight: 0.002987716579809785, Index: 474

Top 3 weights for A2_sm_0:
Weight: 0.024739468470215797, Index: 608
Weight: 0.023569218814373016, Index: 609
Weight: 0.022154532372951508, Index: 4
Weight: 0.018230421468615532, Index: 610
Weight: 0.012074904516339302, Index: 5

Top 3 weights for A3_sm_0:
Weight: 0.005349279846996069, Index: 348
Weight: 0.0037194518372416496, Index: 365
Weight: 0.0034006121568381786, Index: 301
Weight: 0.003209332935512066, Index: 504
Weight: 0.0031813830137252808, Index: 178

A4_sm_0: tensor([[0.4311, 0.4505, 0.1184]])



In [15]:
thr_raw_0 = node0['b1'] / node0['t1']
thr_fft_0 = node0['b2'] / node0['t2']
thr_der_0 = node0['b3'] / node0['t3']

ur_0 = thr_raw_0[:, 87]
us_0 = thr_fft_0[:, 608]
ud_0 = thr_der_0[:, 348]

print("ur_0:", ur_0)
print("us_0:", us_0)
print("ud_0:", ud_0)

ur_0: tensor([4.2870])
us_0: tensor([0.1798])
ud_0: tensor([-0.2676])


In [ ]:
thr__0[:,6]

Plotting rules

In [28]:
df_signal = pd.DataFrame(Xder)
df_signal["label"] = ytrain
df_signal["signal_id"] = df_signal.index

# Melt the dataframe: each row represents a time point of a signal.
df_long = df_signal.melt(id_vars=["signal_id", "label"], var_name="Time", value_name="Amplitude")

# Create the line plot with signals grouped by signal_id and colored by label.
fig = px.line(df_long, x="Time", y="Amplitude", color="label", line_group="signal_id",
              title="Signals of Xder Colored by Label")
fig.add_hline(y=-0.2676, line_color="black")
fig.show()